# Task 1.1 — Core Contribution / Architecture
**Paper:** *Infinite SVM: A Dirichlet Process Mixture of Large-margin Kernel Machines*  
**Authors:** Jun Zhu, Ning Chen, Eric P. Xing  
**Venue:** ICML 2011


## Step-by-Step Method Description

---

### Step 1: Input Representation and Problem Setup
- **Description:** Each input sample **x** ∈ ℝ^M is a feature vector; the response variable Y takes values in {1, …, L} (multi-way classification). A discriminant function F(y, **x**; η) parameterised by η drives prediction.
- **Reference:** Section 2.1, Equation (2)
- **Purpose:** Establishes the notation and the large-margin prediction rule y* = arg max_y E_q(η)[F(y, **x**; η)].

---

### Step 2: Dirichlet Process Prior over Classifiers (Stick-Breaking)
- **Description:** Rather than placing a fixed Gaussian prior on η, iSVM places a DP prior p₀(η) ∼ DP(G₀, α) over the space of classifier parameters. The stick-breaking construction generates a countably infinite set of components with mixing weights π_i(v) = v_i ∏_{j<i}(1−v_j), where v_i ~ Beta(1, α).
- **Reference:** Section 2.3, stick-breaking process enumeration (steps 1–3a)
- **Purpose:** Allows the number of active experts to grow with data, eliminating the need to fix K in advance—this is the core Bayesian nonparametric contribution.

---

### Step 3: Component Assignment via Latent Variable Z
- **Description:** Each data point **x**_d is assigned to a component z_d drawn from Mult(π(**v**)). The component-wise discriminant function is then F(y, **x**; z, η) = η_z^⊤ f(y, **x**), where f(y, **x**) is an ML-dimensional indicator vector (Eq. 3).
- **Reference:** Section 2.3, Equation (3), Figure 2 (graphical model)
- **Purpose:** Enables each data point to be classified by its most relevant local expert instead of a single global classifier.

---

### Step 4: Hybrid Learning Objective (MED + Variational DP)
- **Description:** Two objectives are combined in Equation (8): (a) the MED large-margin objective KL(q(z, η) ∥ p₀(z, η)) + C₁ R(q(z, η)), which enforces large margin without requiring a normalised likelihood; and (b) the variational KL for the DP feature mixture C₂ KL(q(z, γ, v) ∥ p(z, γ, v | D)).
- **Reference:** Section 2.3, Equation (8); Section 2.1, Equation (1)
- **Purpose:** Couples the mixture-of-classifiers with the mixture-of-feature-distributions so that components that explain the input geometry also make accurate predictions.

---

### Step 5: Coordinate Descent with Mean-Field Approximation
- **Description:** The full posterior is approximated with a truncated mean-field: q(z, η, γ, v) = ∏_d q(z_d) ∏_{t≤T} q(η_t) q(γ_t) ∏_{t<T} q(v_t), where T is a finite truncation level and q(v_T = 1) = 1. The algorithm alternates between updating q(η), q(z), q(v), q(γ).
- **Reference:** Section 3, opening paragraph; T is the variational truncation parameter.
- **Purpose:** Makes inference tractable while providing a provable upper bound on the true KL objective.

---

### Step 6: Update q(η_t) → Yields a Cost-Sensitive SVM (Efficient Relaxation)
- **Description:** Maximising the Lagrangian over q(η_t) under a Gaussian base distribution G₀ = N(0, I) gives q(η_t) ~ N(μ_t, I) with shifted mean μ_t = Σ₀ Σ_d ϕ_d^t Σ_y ω_d^y f_d^Δ(y) (Eq. 9). Substituting into the Lagrangian yields the dual QP of a multi-class, cost-sensitive SVM (Eq. 12) with per-sample weights ϕ_d^t.
- **Reference:** Section 3 and Section 3.1, Equations (9), (10), (12)
- **Purpose:** Each component reduces to a standard weighted SVM—training leverages any high-performance SVM solver.

---

### Step 7: Update q(z_d) → Variational Assignment Probabilities φ
- **Description:** Setting ∂L/∂ϕ_d^t = 0 gives the soft assignment update (Eq. 11):  
  ϕ_d^t ∝ exp{ E[log v_t] + Σ_{i<t} E[log(1−v_i)] + ρ · (E[γ_t]^⊤ x_d − E[A(γ_t)]) + (1−ρ) · Σ_y ω_d^y μ_t^⊤ f_d^Δ(y) },  
  where ρ = C₂/(1 + C₂). This couples the feature distribution and the large-margin classifier.
- **Reference:** Section 3, Equation (11)
- **Purpose:** Allows data points near a decision boundary to migrate to a component where they are better classified, creating competing local experts.

---

### Step 8: Update q(v_t) and q(γ_t) — Standard DP Variational Updates
- **Description:** The stick-breaking variational parameters follow the standard Blei & Jordan (2006) DP-VI closed forms: ν_{t,1} = 1 + Σ_d ϕ_d^t, ν_{t,2} = α + Σ_d Σ_{i>t} ϕ_d^i. Feature parameters q(γ_t) are updated similarly using sufficient statistics weighted by ϕ.
- **Reference:** Section 3, citing Blei & Jordan (2006); q(v_t) = Beta(ν_{t,1}, ν_{t,2})
- **Purpose:** Maintains the DP nonparametric structure that controls how many components are active.

---

### Step 9: Kernel Extension
- **Description:** Because the prediction rule (Eq. 5) depends only on the dot-product μ_t^⊤ f(y, **x**), and the SVM dual (Eq. 10) depends only on dot-products of data, a kernel function k(**x**, **x**') can be substituted for any inner product, yielding RBF-iSVM.
- **Reference:** Section 3, last paragraph of Section 3.1
- **Purpose:** Enables local nonlinear classifiers within each component without additional algorithm changes.

---

### Step 10: Test-Time Inference
- **Description:** For a new point **x**, the component assignment q(z|**x**, D) is inferred by one iteration of variational minimisation of KL(q(z) ∥ p(z|**x**, D)) (Eq. 13), holding q(v, γ) fixed at the training posterior. Prediction follows the rule in Eq. (5), approximated by selecting the single most-probable component.
- **Reference:** Section 3.2, Equation (13)
- **Purpose:** Provides an inductive, fully probabilistic classification that accounts for which cluster the new point belongs to.

---

## Final Summary Sentence

This paper solves multi-way classification for complex, heterogeneous data by developing Infinite SVM (iSVM)—a Dirichlet process mixture of large-margin kernel SVMs that automatically determines the number of experts; the authors claim their approach outperforms both global SVMs and DP mixtures of generalized linear models because it avoids the need for a normalised likelihood while still capturing local nonlinearity through kernel-based large-margin classifiers within each mixture component.
